In [1]:
import librosa
from IPython import display as disp

import src.synthesizer as synthesizer

/home/grads/q/quamer.waris/anaconda3/envs/streamVC/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [allow_tf32, disable_jit_profiling]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
/mnt/nvme-data1/waris/PSI-TAMU/streamVC/anonymization/demo_speaker_embeddings.py:5: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier


In [2]:
config = "experiments/base/config.json"
ckpt = "experiments/base/g_01120000"
device = "cuda"

In [3]:
synthesizer.load_models(config, ckpt, device)

/home/grads/q/quamer.waris/anaconda3/envs/streamVC/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/mnt/nvme-data1/waris/PSI-TAMU/streamVC/src/utils.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recom

Loading 'experiments/base/g_01120000'
Complete.
Loaded Encoder and Generator models.


INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
/home/grads/q/quamer.waris/anaconda3/envs/streamVC/lib/python3.10/site-packages/speechbrain/utils/autocast.py:68: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not

Loaded Speaker encoder.
Model type of anonymizer: gan


INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding

Loaded Anonymizer.


/mnt/nvme-data1/waris/PSI-TAMU/streamVC/anonymization/demo_gan_anonymizer.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.gan_vectors = torch.load(model_dir / self

In [4]:
wav_path = "data/speakers/14/wav/14_208_000001_000000.wav"
ref_wav_path = "data/speakers/16/wav/16_122827_000000_000000.wav"

In [5]:
or_audio, _ = librosa.load(wav_path, sr=16000)

disp.Audio(or_audio, rate=16000)

In [6]:
ref_audio, _ = librosa.load(ref_wav_path, sr=16000)

disp.Audio(ref_audio, rate=16000)

### Voice Conversion

In [7]:
gen_audio, rtf_, e2e_latency_ = synthesizer.process_wav_stream(wav_path, ref_wav_path, chunk_size=160)
print(f"RTF: {rtf_}, E2E Latency: {e2e_latency_}")

RTF: 0.16747192855466878, E2E Latency: 186.79550856874698


In [8]:
disp.Audio(gen_audio, rate=16000)

### Anonymization

In [9]:
gen_audio, rtf_, e2e_latency_ = synthesizer.process_wav_stream(wav_path, chunk_size=160)
print(f"RTF: {rtf_}, E2E Latency: {e2e_latency_}")

RTF: 0.16509030471768293, E2E Latency: 186.41444875482927


In [10]:
disp.Audio(gen_audio, rate=16000)